# CacheBlend All-in-One Colab Notebook

This notebook is self-contained. It includes:
- KV pack/unpack + store
- Adaptive token selector (paper-style thresholds)
- Selective recompute engine
- Blend operation (PyTorch fallback)
- End-to-end CacheBlend pipeline
- Direct smoke tests and a quick TTFT benchmark

Run cells in order.

In [ ]:
# Run once in Colab
!pip install -U pip
!pip install "transformers>=4.45,<5" datasets evaluate rouge_score sentence-transformers bitsandbytes accelerate ninja pytest tqdm

# Optional but recommended for faster HF downloads (set your token in Colab secrets)
# import os
# os.environ["HF_TOKEN"] = "<your_token_here>"

In [ ]:
import json
import time
from statistics import mean, pstdev
from typing import Dict, List, Optional

import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer


# -----------------------------
# KV helpers + cache store
# -----------------------------
def pack_kv(past_key_values) -> torch.Tensor:
    """HuggingFace cache -> (L, 2, N, H, D)."""
    # DynamicCache-style API
    if hasattr(past_key_values, "key_cache") and hasattr(past_key_values, "value_cache"):
        layers = []
        for k, v in zip(past_key_values.key_cache, past_key_values.value_cache):
            k = k.squeeze(0).transpose(0, 1)  # (N, H, D)
            v = v.squeeze(0).transpose(0, 1)
            layers.append(torch.stack([k, v], dim=0))
        return torch.stack(layers, dim=0)

    # Some cache classes expose a legacy tuple-of-tuples view.
    if hasattr(past_key_values, "to_legacy_cache"):
        past_key_values = past_key_values.to_legacy_cache()

    layers = []
    for layer_kv in past_key_values:
        if not isinstance(layer_kv, (tuple, list)) or len(layer_kv) < 2:
            raise ValueError("Unsupported cache layer format in past_key_values.")

        # Newer HF variants may include extra entries per layer.
        k, v = layer_kv[0], layer_kv[1]
        k = k.squeeze(0).transpose(0, 1)
        v = v.squeeze(0).transpose(0, 1)
        layers.append(torch.stack([k, v], dim=0))

    return torch.stack(layers, dim=0)


def unpack_kv(kv_tensor: torch.Tensor):
    """(L, 2, N, H, D) -> DynamicCache."""
    from transformers import DynamicCache

    cache = DynamicCache()
    for i in range(kv_tensor.shape[0]):
        k = kv_tensor[i, 0].permute(1, 0, 2).unsqueeze(0).half()  # (1, H, N, D)
        v = kv_tensor[i, 1].permute(1, 0, 2).unsqueeze(0).half()
        cache.update(k, v, i)
    return cache


class KVCacheStore:
    def __init__(self):
        self._data = {}

    def store(self, key: str, kv_tensor: torch.Tensor):
        self._data[key] = kv_tensor.cpu().clone()

    def load(self, key: str, device: str = "cuda") -> Optional[torch.Tensor]:
        kv = self._data.get(key)
        if kv is None:
            return None
        return kv.to(device)


# -----------------------------
# Adaptive selector (paper-style)
# -----------------------------
class AdaptiveTokenSelector:
    def __init__(
        self,
        model,
        low_thresh: float = 0.05,
        high_thresh: float = 0.20,
        min_k_ratio: float = 0.05,
        max_k_ratio: float = 0.30,
        eps: float = 1e-8,
    ):
        self.model = model
        self.low_thresh = low_thresh
        self.high_thresh = high_thresh
        self.min_k_ratio = min_k_ratio
        self.max_k_ratio = max_k_ratio
        self.eps = eps
        self.last_selection = {}
        self.history: List[Dict[str, float]] = []
        self.model.eval()

    def _project_cached_for_cosine(self, cached_k_mean: torch.Tensor, target_dim: int) -> torch.Tensor:
        cached_dim = cached_k_mean.shape[-1]
        if cached_dim == target_dim:
            return cached_k_mean
        if cached_dim > target_dim:
            return cached_k_mean[:, :target_dim]
        return F.pad(cached_k_mean, (0, target_dim - cached_dim), mode="constant", value=0.0)

    def _adaptive_ratio(self, mean_divergence: float) -> float:
        if mean_divergence <= self.low_thresh:
            return self.min_k_ratio
        if mean_divergence >= self.high_thresh:
            return self.max_k_ratio
        alpha = (mean_divergence - self.low_thresh) / max(self.high_thresh - self.low_thresh, self.eps)
        return self.min_k_ratio + alpha * (self.max_k_ratio - self.min_k_ratio)

    def select(self, chunk_ids: torch.Tensor, cached_kv: torch.Tensor) -> torch.Tensor:
        # We have no padding here, so an all-ones mask is appropriate.
        attn_mask = torch.ones_like(chunk_ids, device=chunk_ids.device)

        with torch.no_grad():
            out = self.model(
                chunk_ids,
                attention_mask=attn_mask,
                use_cache=True,
                output_hidden_states=True,
                return_dict=True,
            )
            fresh_hidden = out.hidden_states[1].squeeze(0).float()  # (N, d_model)

        cached_k = cached_kv[0, 0].float()  # (N, H, D)
        cached_hidden = cached_k.mean(dim=1)  # (N, D)
        cached_hidden = self._project_cached_for_cosine(cached_hidden, fresh_hidden.shape[-1])

        fresh_norm = F.normalize(fresh_hidden + self.eps, p=2, dim=-1)
        cached_norm = F.normalize(cached_hidden + self.eps, p=2, dim=-1)

        cosine = F.cosine_similarity(fresh_norm, cached_norm, dim=-1)
        divergence = (1.0 - cosine).clamp_min(0.0)

        mean_div = float(divergence.mean().item())
        ratio = self._adaptive_ratio(mean_div)

        n_tokens = int(chunk_ids.shape[1])
        k = min(n_tokens, max(1, int(ratio * n_tokens)))
        indices = torch.topk(divergence, k, largest=True, sorted=False).indices
        indices = torch.sort(indices).values.to(torch.int64)

        self.last_selection = {
            "sequence_length": float(n_tokens),
            "mean_divergence": mean_div,
            "selected_k_ratio": float(ratio),
            "selected_k": float(k),
        }
        self.history.append(self.last_selection.copy())

        return indices


# -----------------------------
# Recompute + blend + pipeline
# -----------------------------
class SelectiveRecomputer:
    def __init__(self, model):
        self.model = model

    def recompute(self, chunk_ids: torch.Tensor, cached_kv: torch.Tensor, indices: torch.Tensor) -> torch.Tensor:
        selected_ids = chunk_ids[:, indices]  # (1, k)
        past = unpack_kv(cached_kv)

        with torch.no_grad():
            out = self.model(selected_ids, past_key_values=past, use_cache=True)

        new_kv_full = pack_kv(out.past_key_values)  # (L, 2, N+k, H, D)
        k = indices.shape[0]
        return new_kv_full[:, :, -k:, :, :]


class KVBlender:
    def blend(self, cached_kv: torch.Tensor, new_values: torch.Tensor, indices: torch.Tensor) -> torch.Tensor:
        cached_kv[:, :, indices, :, :] = new_values
        return cached_kv


def cacheblend_generate(
    prompt,
    chunk_texts,
    model,
    tokenizer,
    store,
    selector,
    recomputer,
    blender=None,
    mode: str = "cacheblend",  # "standard_cache" or "cacheblend"
    max_new_tokens=32,
    min_new_tokens=8,
    do_sample=True,
    temperature=0.8,
    top_p=0.95,
    benchmark_first_token: bool = False,
):
    if blender is None:
        blender = KVBlender()

    cache_hits, cache_misses = 0, 0
    fused_kv = None

    # End-to-end TTFT should include chunk processing + first-token generation.
    torch.cuda.synchronize()
    t_request_start = time.perf_counter()

    for chunk_text in chunk_texts:
        chunk_enc = tokenizer(
            chunk_text,
            return_tensors="pt",
            return_attention_mask=True,
            truncation=True,
            max_length=512,
        )
        chunk_ids = chunk_enc["input_ids"].to("cuda")
        chunk_attn = chunk_enc["attention_mask"].to("cuda")

        cached = store.load(chunk_text, device="cuda")

        if cached is None:
            cache_misses += 1
            with torch.no_grad():
                out = model(chunk_ids, attention_mask=chunk_attn, use_cache=True)
            chunk_kv = pack_kv(out.past_key_values)
            store.store(chunk_text, chunk_kv)
        else:
            cache_hits += 1
            chunk_kv = cached

            if mode == "cacheblend":
                indices = selector.select(chunk_ids, chunk_kv)
                new_kv = recomputer.recompute(chunk_ids, chunk_kv, indices)
                chunk_kv = blender.blend(chunk_kv, new_kv, indices)
            elif mode == "standard_cache":
                # Reuse cached KV directly.
                pass
            else:
                raise ValueError(f"Unsupported mode: {mode}")

        fused_kv = chunk_kv if fused_kv is None else torch.cat([fused_kv, chunk_kv], dim=2)

    prompt_enc = tokenizer(
        prompt,
        return_tensors="pt",
        return_attention_mask=True,
        truncation=True,
        max_length=128,
    )
    prompt_ids = prompt_enc["input_ids"].to("cuda")
    prompt_attn = prompt_enc["attention_mask"].to("cuda")

    past = unpack_kv(fused_kv) if fused_kv is not None else None

    torch.cuda.synchronize()
    t_decode_start = time.perf_counter()

    context_len = fused_kv.shape[2] if fused_kv is not None else 0
    cache_position = torch.arange(context_len, context_len + prompt_ids.shape[1], device=prompt_ids.device)

    gen_max = 1 if benchmark_first_token else max_new_tokens
    gen_min = 1 if benchmark_first_token else min_new_tokens
    gen_sample = False if benchmark_first_token else do_sample

    with torch.no_grad():
        out_ids = model.generate(
            prompt_ids,
            attention_mask=prompt_attn,
            past_key_values=past,
            cache_position=cache_position,
            max_new_tokens=gen_max,
            min_new_tokens=gen_min,
            use_cache=True,
            do_sample=gen_sample,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    torch.cuda.synchronize()
    decode_only_ms = (time.perf_counter() - t_decode_start) * 1000
    e2e_ttft_ms = (time.perf_counter() - t_request_start) * 1000

    generated = out_ids[0][prompt_ids.shape[1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True)

    return {
        "text": text,
        "ttft_ms": e2e_ttft_ms,
        "e2e_ttft_ms": e2e_ttft_ms,
        "generate_only_ms": decode_only_ms,
        "cache_hits": cache_hits,
        "cache_misses": cache_misses,
    }


def run_three_mode_benchmark(
    prompt,
    chunk_texts,
    model,
    tokenizer,
    trials: int = 5,
    warmup_runs: int = 1,
):
    cold_full = []
    warm_standard = []
    warm_cacheblend = []
    per_trial = []

    for _ in range(warmup_runs):
        s = KVCacheStore()
        sel = AdaptiveTokenSelector(model=model, low_thresh=0.2, high_thresh=1.1)
        rec = SelectiveRecomputer(model)
        bl = KVBlender()
        _ = cacheblend_generate(prompt, chunk_texts, model, tokenizer, s, sel, rec, bl, mode="standard_cache", benchmark_first_token=True)
        _ = cacheblend_generate(prompt, chunk_texts, model, tokenizer, s, sel, rec, bl, mode="cacheblend", benchmark_first_token=True)

    for t in range(trials):
        # 1) Cold full prefill request
        store_cold = KVCacheStore()
        sel_cold = AdaptiveTokenSelector(model=model, low_thresh=0.2, high_thresh=1.1)
        rec_cold = SelectiveRecomputer(model)
        bl_cold = KVBlender()
        cold = cacheblend_generate(
            prompt,
            chunk_texts,
            model,
            tokenizer,
            store_cold,
            sel_cold,
            rec_cold,
            bl_cold,
            mode="standard_cache",
            benchmark_first_token=True,
        )

        # 2) Warm standard KV cache request (no recompute)
        store_std = KVCacheStore()
        sel_std = AdaptiveTokenSelector(model=model, low_thresh=0.2, high_thresh=1.1)
        rec_std = SelectiveRecomputer(model)
        bl_std = KVBlender()
        _ = cacheblend_generate(
            prompt,
            chunk_texts,
            model,
            tokenizer,
            store_std,
            sel_std,
            rec_std,
            bl_std,
            mode="standard_cache",
            benchmark_first_token=True,
        )
        warm_std = cacheblend_generate(
            prompt,
            chunk_texts,
            model,
            tokenizer,
            store_std,
            sel_std,
            rec_std,
            bl_std,
            mode="standard_cache",
            benchmark_first_token=True,
        )

        # 3) Warm CacheBlend request (adaptive recompute + blend)
        store_cb = KVCacheStore()
        sel_cb = AdaptiveTokenSelector(model=model, low_thresh=0.2, high_thresh=1.1)
        rec_cb = SelectiveRecomputer(model)
        bl_cb = KVBlender()
        _ = cacheblend_generate(
            prompt,
            chunk_texts,
            model,
            tokenizer,
            store_cb,
            sel_cb,
            rec_cb,
            bl_cb,
            mode="cacheblend",
            benchmark_first_token=True,
        )
        warm_cb = cacheblend_generate(
            prompt,
            chunk_texts,
            model,
            tokenizer,
            store_cb,
            sel_cb,
            rec_cb,
            bl_cb,
            mode="cacheblend",
            benchmark_first_token=True,
        )

        cold_full.append(cold["e2e_ttft_ms"])
        warm_standard.append(warm_std["e2e_ttft_ms"])
        warm_cacheblend.append(warm_cb["e2e_ttft_ms"])

        per_trial.append(
            {
                "trial": t + 1,
                "cold_full_ttft_ms": cold["e2e_ttft_ms"],
                "warm_standard_ttft_ms": warm_std["e2e_ttft_ms"],
                "warm_cacheblend_ttft_ms": warm_cb["e2e_ttft_ms"],
                "speedup_cold_vs_standard": cold["e2e_ttft_ms"] / max(warm_std["e2e_ttft_ms"], 1e-9),
                "speedup_cold_vs_cacheblend": cold["e2e_ttft_ms"] / max(warm_cb["e2e_ttft_ms"], 1e-9),
                "cacheblend_selector_stats": sel_cb.last_selection,
            }
        )

    summary = {
        "trials": trials,
        "warmup_runs": warmup_runs,
        "cold_full_mean_ms": mean(cold_full),
        "cold_full_std_ms": pstdev(cold_full) if len(cold_full) > 1 else 0.0,
        "warm_standard_mean_ms": mean(warm_standard),
        "warm_standard_std_ms": pstdev(warm_standard) if len(warm_standard) > 1 else 0.0,
        "warm_cacheblend_mean_ms": mean(warm_cacheblend),
        "warm_cacheblend_std_ms": pstdev(warm_cacheblend) if len(warm_cacheblend) > 1 else 0.0,
        "speedup_cold_vs_standard_mean": mean([c / max(s, 1e-9) for c, s in zip(cold_full, warm_standard)]),
        "speedup_cold_vs_cacheblend_mean": mean([c / max(cb, 1e-9) for c, cb in zip(cold_full, warm_cacheblend)]),
    }

    return {"summary": summary, "trials": per_trial}


print("Definitions loaded.")

In [ ]:
# Runtime + model load
assert torch.cuda.is_available(), "Please enable GPU runtime in Colab (T4)."

device = "cuda"
# Use GPT-2 for cleaner behavior than tiny-gpt2 while still fitting Colab T4 comfortably.
model_id = "gpt2"

print(f"Loading: {model_id}")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16).to(device).eval()

print("Model loaded.")

In [ ]:
# Direct Colab smoke tests + repeated benchmark (no pytest needed)

# Test 1: adaptive selector output contract
text = (
    "CacheBlend recomputes only highly divergent tokens to reduce prefill latency. "
    "This validates adaptive token selection behavior."
)
chunk_enc = tokenizer(text, return_tensors="pt", return_attention_mask=True)
chunk_ids = chunk_enc["input_ids"].to(device)
chunk_mask = chunk_enc["attention_mask"].to(device)

with torch.no_grad():
    out = model(chunk_ids, attention_mask=chunk_mask, use_cache=True)

cached_kv = pack_kv(out.past_key_values).to(device)
selector = AdaptiveTokenSelector(model=model)
indices = selector.select(chunk_ids, cached_kv)
stats = selector.last_selection

assert indices.dtype == torch.int64
assert indices.device.type == "cuda"
assert indices.numel() >= 1
assert torch.all(indices[1:] >= indices[:-1]), "Indices must be sorted"
assert 0.05 <= stats["selected_k_ratio"] <= 0.30
print("Test 1 passed:", stats)


# Test 2: end-to-end cold/warm run
store = KVCacheStore()
recomputer = SelectiveRecomputer(model)
blender = KVBlender()

chunks = [
    "Paris is the capital of France and has many famous landmarks. " * 8,
    "The Eiffel Tower is one of the most visited monuments in the world. " * 8,
    "CacheBlend reuses cached KV and selectively recomputes high-divergence tokens. " * 8,
    "Selective recomputation reduces full prefill cost when cached context is reusable. " * 8,
]
prompt = "Summarize how CacheBlend can reduce latency for repeated context queries."

cold = cacheblend_generate(
    prompt, chunks, model, tokenizer, store, selector, recomputer, blender, max_new_tokens=24
)
warm = cacheblend_generate(
    prompt, chunks, model, tokenizer, store, selector, recomputer, blender, max_new_tokens=24
)

assert cold["cache_misses"] == len(chunks)
assert warm["cache_hits"] == len(chunks)
assert len(warm["text"]) > 0
assert warm["ttft_ms"] > 0

print("Test 2 passed")
print(f"COLD TTFT: {cold['ttft_ms']:.2f} ms")
print(f"WARM TTFT: {warm['ttft_ms']:.2f} ms")
print(f"Speedup: {cold['ttft_ms']/warm['ttft_ms']:.2f}x")
print("Output:", warm["text"][:200])
print("Output repr:", repr(warm["text"][:200]))


# Test 3: repeated benchmark for stable report numbers
bench = run_three_mode_benchmark(
    prompt=prompt,
    chunk_texts=chunks,
    model=model,
    tokenizer=tokenizer,
    trials=5,
    warmup_runs=1,
)

print("\nBenchmark summary:")
for k, v in bench["summary"].items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

with open("cacheblend_colab_results.json", "w") as f:
    json.dump(bench, f, indent=2)

print("Saved benchmark JSON -> cacheblend_colab_results.json")